# The top of the raw table is not medicine

One FAERS partition, 12,000 reports. Before any statistic, the
question is what the reaction field actually contains.

In [1]:
import duckdb

PARTITION = "../data/parquet/year=2025/quarter=1/part=0001-of-0028"

drugs = f"'{PARTITION}/report_drug.parquet'"
reactions = f"'{PARTITION}/report_reaction.parquet'"
reports = f"'{PARTITION}/report.parquet'"

In [2]:
top_terms = f"""
    SELECT reactionmeddrapt AS term, count(*) AS rows
    FROM {reactions}
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 12
"""

duckdb.sql(top_terms)

┌──────────────────────────────────────┬───────┐
│                 term                 │ rows  │
│               varchar                │ int64 │
├──────────────────────────────────────┼───────┤
│ Off label use                        │   992 │
│ Drug ineffective                     │   756 │
│ Fatigue                              │   581 │
│ Diarrhoea                            │   484 │
│ Product dose omission issue          │   473 │
│ Nausea                               │   459 │
│ Headache                             │   421 │
│ Pruritus                             │   410 │
│ Dyspnoea                             │   403 │
│ Rash                                 │   345 │
│ Condition aggravated                 │   336 │
│ Product use in unapproved indication │   336 │
└──────────────────────────────────────┴───────┘
  12 rows                            2 columns

`Off label use`, `Product use in unapproved indication`,
`Drug ineffective` — none of these is something that happened to a
patient. They record how a product was used, or whether it worked.
Ranked by disproportionality they would sit at the top of every
table and mean nothing.

The exclusion list is the response: a committed CSV, one reason per
row, reviewed each milestone.

In [3]:
excluded = duckdb.sql("""
    SELECT term, category, reason
    FROM read_csv('../reference/excluded_terms.csv', comment='#')
""").df()

len(excluded), excluded.category.value_counts().to_dict()

(187,
 {'administration': 55,
  'product_quality': 37,
  'device': 28,
  'supply': 24,
  'exposure': 21,
  'efficacy': 9,
  'indication': 7,
  'therapy_decision': 5,
  'no_event': 1})

The `comment='#'` above is load-bearing. The file opens with a prose
header explaining the membership rule, and a read without that flag
returns zero rows and no error — the list would silently stop
existing. `analysis/prr.py` raises on an empty read for that reason.

In [4]:
after = f"""
    SELECT reactionmeddrapt AS term, count(*) AS rows
    FROM {reactions}
    WHERE reactionmeddrapt NOT IN (
        SELECT term FROM read_csv('../reference/excluded_terms.csv', comment='#')
    )
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 12
"""

duckdb.sql(after)

┌────────────┬───────┐
│    term    │ rows  │
│  varchar   │ int64 │
├────────────┼───────┤
│ Fatigue    │   581 │
│ Diarrhoea  │   484 │
│ Nausea     │   459 │
│ Headache   │   421 │
│ Pruritus   │   410 │
│ Dyspnoea   │   403 │
│ Rash       │   345 │
│ Death      │   329 │
│ Vomiting   │   313 │
│ Pain       │   313 │
│ Arthralgia │   312 │
│ Dizziness  │   287 │
└────────────┴───────┘
  12 rows  2 columns

## 187 terms remove 15.4% of the reaction rows

What is left is symptoms: fatigue, diarrhoea, nausea, headache.
`Death` sits eighth, which is the right shape for a spontaneous
reporting system — serious outcomes are reported, and they are not
the majority of what is reported.

The list is a floor, not an enumeration. Procedure terms
(`Chemotherapy`, `Radiotherapy`) are not bodily responses either,
but excluding them by hand loses: they need the MedDRA hierarchy,
and openFDA ships the preferred term only.

In [5]:
duckdb.sql(f"""
    SELECT
        count(*) AS reaction_rows,
        count(*) FILTER (WHERE reactionmeddrapt IN (
            SELECT term FROM read_csv('../reference/excluded_terms.csv', comment='#')
        )) AS excluded_rows,
        count(DISTINCT reactionmeddrapt) AS distinct_terms
    FROM {reactions}
""")

┌───────────────┬───────────────┬────────────────┐
│ reaction_rows │ excluded_rows │ distinct_terms │
│     int64     │     int64     │     int64      │
├───────────────┼───────────────┼────────────────┤
│         44916 │          6900 │           4281 │
└───────────────┴───────────────┴────────────────┘